## 1. Import packages and set options 
<a name="import_packages"></a>

In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [ ]:
#this is the database we will be extracting from
db1 = "familyman_dev_v3" #database where Familyman data is stored
db2 = "fcsq" #database where tables created as part of FCSQ processing are stored where required

#this extracts the February snapshot from athena
snapshot_date = "2023-02-09"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

### C100 cases - CAO, specific issue, and prohibited steps

In [ ]:
#Getting list of cases with c100 apps
pydb.create_temp_table( 
f"""
SELECT 
  DISTINCT 
    case_number
FROM
  {db2}.ca_apps_order_types
WHERE
  order_code IN (29,30,31,32)

""",

"c100_apps")

In [ ]:
#Getting list of cases with c100 ords
pydb.create_temp_table( 
f"""
SELECT 
  DISTINCT 
    case_number
FROM
  {db2}.ca_all_disposals
WHERE
  order_code IN (29,30,31,32)
  and disp_type_code = 1

""",

"c100_ords")

In [ ]:
#Get a distinct list of case numbers with either a C100 app or order
#Union query should exclude duplicate records
pydb.create_temp_table( 
f"""
SELECT 
    case_number
FROM
  __temp__.c100_apps
UNION
SELECT 
  case_number
FROM
  __temp__.c100_ords

""",

"c100_cases")

In [ ]:
pydb.read_sql_query ("select count (*) as c100 FROM __temp__.c100_cases")

### Appeals

In [ ]:
#Now get a cases that have appeals recorded under the G50 event (APP), or other (OTH) app types are recorded where we can search the free text narrative field  
pydb.create_temp_table( 
f"""
SELECT 
    e.case_number,
    e.receipt_date,
    e.narrative,
    EXTRACT(year FROM e.receipt_date) AS year,
    f.event,
    f.field_model,
    f.value as all_event_app_types,
    TRIM(ord_type) as order_type
  FROM 
    {db1}.event_fields F
    INNER JOIN {db1}.events e
      ON f.event = e.event
    CROSS JOIN UNNEST(SPLIT(f.value,',')) AS t(ord_type)
  WHERE 
    field_model IN('G50_AT')
    AND   TRIM(ord_type) IN ('APP','OTH')
    AND e.error = 'N'
    AND f.mojap_snapshot_date = DATE'{snapshot_date}'
    AND e.mojap_snapshot_date = DATE'{snapshot_date}'
""",

"appeal_events")

In [ ]:
#Now filter the 'OTH' app type to just those that contain the word appeal, and keep only those that were received in 2021
pydb.create_temp_table( 
f"""
SELECT  
    case_number,
    receipt_date,
    field_model,
    order_type,
    narrative
FROM
  __temp__.appeal_events
WHERE
  year = 2021
  AND (order_type = 'APP'
    OR (order_type = 'OTH' AND UPPER(narrative) LIKE '%APPEAL%'))
    
""",

"appeals")

### Matching appeals to C100 cases

In [ ]:
#Now match these to the case numbers that have a C100 app or order to see how many had an appeal recorded in 2021
pydb.read_sql_query(
f"""
SELECT  
    count(*) AS appeals
FROM
  __temp__.appeals
WHERE
  case_number IN (SELECT case_number
                  FROM __temp__.C100_cases)""")
    
